# TabR POCABS Bias-Reduced External Evaluation

Evaluate existing TabR frame-level anxiety models on POCABS datatest, then report frame-, event-, and subject-level metrics.

This notebook **does not train a new model by default**. It uses existing checkpoints and focuses on reducing subject-level class bias by separating event and subject thresholds.

Core flow:

1. load existing TabR models
2. predict frame probabilities on POCABS datatest
3. aggregate frame probabilities into event probabilities
4. calibrate subject aggregation separately from event aggregation
5. export summaries, predictions, and confusion matrices to `output/apex/datatest/pocabs_bias_v1/`

Bias-reduction changes:

- event threshold and subject threshold are separate
- subject aggregation tries `mean`, `median`, `trimmed_mean`, `p90`, `top3_mean`, `proportion`, and `count`
- `proportion` = percentage of events above event threshold
- `count` = number of events above event threshold


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
TABR_ROOT = ROOT / 'third_party' / 'tabular-dl-tabr-official'
os.environ['PROJECT_DIR'] = str(TABR_ROOT)
if str(TABR_ROOT) not in sys.path:
    sys.path.append(str(TABR_ROOT))

import bin.go
from scripts.export_tabr_pocabs_event import EXPERIMENTS, export_all
from scripts.evaluate_tabr_pocabs_frame import OUT_ROOT, evaluate_all

N_EVAL_SEEDS = 15
N_ENSEMBLES = 0
ENSEMBLE_SIZE = 5
display(Markdown(f'ROOT: `{ROOT}`  \nTABR_ROOT: `{TABR_ROOT}`'))

display(Markdown('Existing-model subject-bias-reduced evaluation: event threshold and subject threshold are calibrated separately; subject aggregation tries mean/median/trimmed_mean/p90/top3_mean/proportion/count.'))


ROOT: `/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st`  
TABR_ROOT: `/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/third_party/tabular-dl-tabr-official`

Existing-model subject-bias-reduced evaluation: event threshold and subject threshold are calibrated separately; subject aggregation tries mean/median/trimmed_mean/p90/top3_mean/proportion/count.

## 1. Setup

Imports project paths, existing model evaluator, and output root.

Important: the evaluator uses existing models from `third_party/tabular-dl-tabr-official/exp/tabr/`, not newly trained POCABS event-wide models.


In [2]:
export_results = export_all(overwrite=False)
for item in export_results:
    display(Markdown(f"### `{item['exp_dir']}`"))
    display(pd.DataFrame(item['splits']))
    print('feature_count:', item['feature_count'])
    print('tuning_toml:', item['tuning_toml'])


### `/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/third_party/tabular-dl-tabr-official/exp/tabr/convat_apex_pocabs_event_qholdout_q123_q4_q5_v2`

,split,rows,events,subjects,event_labels,frame_count_min,frame_count_max,frame_count_mean
0,train,1663,1663,56,"{'anxiety_tinggi': 982, 'anxiety_rendah': 681}",3,106,21.565244
1,val,550,550,56,"{'anxiety_tinggi': 322, 'anxiety_rendah': 228}",3,89,20.647273
2,test,550,550,56,"{'anxiety_tinggi': 327, 'anxiety_rendah': 223}",2,87,18.976364


feature_count: 41552
tuning_toml: /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/third_party/tabular-dl-tabr-official/exp/tabr/convat_apex_pocabs_event_qholdout_q123_q4_q5_v2/0-tuning.toml


### `/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/third_party/tabular-dl-tabr-official/exp/tabr/convat_apex_pocabs_event_qwalk_q12_q3_q4_v2`

,split,rows,events,subjects,event_labels,frame_count_min,frame_count_max,frame_count_mean
0,train,1113,1113,56,"{'anxiety_tinggi': 657, 'anxiety_rendah': 456}",3,106,22.328841
1,val,550,550,56,"{'anxiety_tinggi': 325, 'anxiety_rendah': 225}",4,73,20.020000
2,test,550,550,56,"{'anxiety_tinggi': 322, 'anxiety_rendah': 228}",3,89,20.647273


feature_count: 41552
tuning_toml: /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/third_party/tabular-dl-tabr-official/exp/tabr/convat_apex_pocabs_event_qwalk_q12_q3_q4_v2/0-tuning.toml


### `/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/third_party/tabular-dl-tabr-official/exp/tabr/convat_apex_pocabs_event_qwalk_q123_q4_q5_v2`

,split,rows,events,subjects,event_labels,frame_count_min,frame_count_max,frame_count_mean
0,train,1663,1663,56,"{'anxiety_tinggi': 982, 'anxiety_rendah': 681}",3,106,21.565244
1,val,550,550,56,"{'anxiety_tinggi': 322, 'anxiety_rendah': 228}",3,89,20.647273
2,test,550,550,56,"{'anxiety_tinggi': 327, 'anxiety_rendah': 223}",2,87,18.976364


feature_count: 41552
tuning_toml: /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/third_party/tabular-dl-tabr-official/exp/tabr/convat_apex_pocabs_event_qwalk_q123_q4_q5_v2/0-tuning.toml


## 2. Optional legacy training cell

This cell is kept only for legacy event-wide experiments. It is **not required** for the bias-reduced evaluation below.

Skip this section unless you intentionally want to resume old POCABS event-wide training.


In [3]:
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

'0'

In [4]:
for cfg in EXPERIMENTS:
    tuning_toml = TABR_ROOT / 'exp' / 'tabr' / cfg['exp_name'] / '0-tuning.toml'
    display(Markdown(f"## Training `{cfg['exp_name']}`"))
    bin.go.main(
        tuning_toml,
        n_seeds=N_EVAL_SEEDS,
        n_ensembles=N_ENSEMBLES,
        ensemble_size=ENSEMBLE_SIZE,
        continue_=True,
        force=False,
    )


## Training `convat_apex_pocabs_event_qholdout_q123_q4_q5_v2`

2026-05-07 01:04:41.320 | INFO     | lib.util:start:124 - Already done!

2026-05-07 01:04:41.328 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.331 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.332 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.335 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.337 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.338 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.340 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.342 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.344 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.345 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.347 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.349 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.351 | WARNING  | li

[>>>] exp/tabr/convat_apex_pocabs_event_qholdout_q123_q4_q5_v2/0-tuning | 2026-05-07 01:04:41.320173
[>>>] exp/tabr/convat_apex_pocabs_event_qholdout_q123_q4_q5_v2/0-evaluation/0 | 2026-05-07 01:04:41.328900
[>>>] exp/tabr/convat_apex_pocabs_event_qholdout_q123_q4_q5_v2/0-evaluation/1 | 2026-05-07 01:04:41.330958
[>>>] exp/tabr/convat_apex_pocabs_event_qholdout_q123_q4_q5_v2/0-evaluation/2 | 2026-05-07 01:04:41.332904
[>>>] exp/tabr/convat_apex_pocabs_event_qholdout_q123_q4_q5_v2/0-evaluation/3 | 2026-05-07 01:04:41.335002
[>>>] exp/tabr/convat_apex_pocabs_event_qholdout_q123_q4_q5_v2/0-evaluation/4 | 2026-05-07 01:04:41.337062
[>>>] exp/tabr/convat_apex_pocabs_event_qholdout_q123_q4_q5_v2/0-evaluation/5 | 2026-05-07 01:04:41.338796
[>>>] exp/tabr/convat_apex_pocabs_event_qholdout_q123_q4_q5_v2/0-evaluation/6 | 2026-05-07 01:04:41.340504
[>>>] exp/tabr/convat_apex_pocabs_event_qholdout_q123_q4_q5_v2/0-evaluation/7 | 2026-05-07 01:04:41.342067
[>>>] exp/tabr/convat_apex_pocabs_event_qho

## Training `convat_apex_pocabs_event_qwalk_q12_q3_q4_v2`

2026-05-07 01:04:41.359 | INFO     | lib.util:start:124 - Already done!

2026-05-07 01:04:41.362 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.364 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.366 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.369 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.370 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.372 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.374 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.376 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.377 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.379 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.380 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.382 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.383 | WARNING  | li

[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q12_q3_q4_v2/0-tuning | 2026-05-07 01:04:41.359680
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q12_q3_q4_v2/0-evaluation/0 | 2026-05-07 01:04:41.362840
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q12_q3_q4_v2/0-evaluation/1 | 2026-05-07 01:04:41.364520
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q12_q3_q4_v2/0-evaluation/2 | 2026-05-07 01:04:41.366748
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q12_q3_q4_v2/0-evaluation/3 | 2026-05-07 01:04:41.368952
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q12_q3_q4_v2/0-evaluation/4 | 2026-05-07 01:04:41.370445
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q12_q3_q4_v2/0-evaluation/5 | 2026-05-07 01:04:41.372696
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q12_q3_q4_v2/0-evaluation/6 | 2026-05-07 01:04:41.374500
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q12_q3_q4_v2/0-evaluation/7 | 2026-05-07 01:04:41.376268
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q12_q3_q4_v2/0-evaluation/8 | 202

## Training `convat_apex_pocabs_event_qwalk_q123_q4_q5_v2`

2026-05-07 01:04:41.390 | INFO     | lib.util:start:124 - Already done!

2026-05-07 01:04:41.397 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.399 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.401 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.402 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.404 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.405 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.406 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.408 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.409 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.410 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.412 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.413 | WARNING  | lib.util:start:120 - Already exists!
2026-05-07 01:04:41.415 | WARNING  | li

[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q123_q4_q5_v2/0-tuning | 2026-05-07 01:04:41.390620
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q123_q4_q5_v2/0-evaluation/0 | 2026-05-07 01:04:41.397590
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q123_q4_q5_v2/0-evaluation/1 | 2026-05-07 01:04:41.399787
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q123_q4_q5_v2/0-evaluation/2 | 2026-05-07 01:04:41.401602
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q123_q4_q5_v2/0-evaluation/3 | 2026-05-07 01:04:41.402950
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q123_q4_q5_v2/0-evaluation/4 | 2026-05-07 01:04:41.404243
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q123_q4_q5_v2/0-evaluation/5 | 2026-05-07 01:04:41.405626
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q123_q4_q5_v2/0-evaluation/6 | 2026-05-07 01:04:41.406924
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q123_q4_q5_v2/0-evaluation/7 | 2026-05-07 01:04:41.408173
[>>>] exp/tabr/convat_apex_pocabs_event_qwalk_q123_q4_q5_v2/0-evaluati

## 3. Bias-reduced external datatest evaluation

Runs existing-model evaluation on POCABS datatest.

Outputs:

- `calibration`: internal event-level calibration per model
- `frame_summary`: frame-level external metrics
- `event_summary`: event-level external metrics
- `subject_summary`: subject-level external metrics with separate subject threshold/rule
- `detail_files`: paths to prediction CSV files

Main output folder: `output/apex/datatest/pocabs_bias_v1/`


In [5]:
outputs = evaluate_all(return_predictions=False)
display(Markdown(f'Results saved to: `{OUT_ROOT}`'))


2026-05-07 01:04:41.695 | INFO     | lib.data:build_dataset:338 - Using cached dataset: build_dataset__convat_apex_anxiety_qholdout_q123_q4_q5_alis__None__None__None__None__72__d10c1508d8934c1de65e6394f780722e.pickle
2026-05-07 01:07:52.177 | INFO     | lib.data:build_dataset:338 - Using cached dataset: build_dataset__convat_apex_anxiety_qwalk_q12_q3_q4_alis__None__None__None__None__72__d10c1508d8934c1de65e6394f780722e.pickle
2026-05-07 01:11:02.986 | INFO     | lib.data:build_dataset:338 - Using cached dataset: build_dataset__convat_apex_anxiety_qwalk_q123_q4_q5_alis__None__None__None__None__72__d10c1508d8934c1de65e6394f780722e.pickle


Results saved to: `/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/output/apex/datatest/pocabs_bias_v1`

In [6]:
csv_files = pd.DataFrame({'file': [p.name for p in sorted(OUT_ROOT.glob('*.csv'))], 'path': [str(p) for p in sorted(OUT_ROOT.glob('*.csv'))], 'size_mb': [p.stat().st_size / 1024 / 1024 for p in sorted(OUT_ROOT.glob('*.csv'))]})
display(csv_files)


,file,path,size_mb
0,calibration.csv,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.001146
1,detail_files.csv,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.005766
2,event_predictions__convat_apex_anxiety_qholdou...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.868990
3,event_predictions__convat_apex_anxiety_qholdou...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.464824
4,event_predictions__convat_apex_anxiety_qholdou...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.397228
5,event_predictions__convat_apex_anxiety_qwalk_q...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.858290
6,event_predictions__convat_apex_anxiety_qwalk_q...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.459102
7,event_predictions__convat_apex_anxiety_qwalk_q...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.392250
8,event_predictions__convat_apex_anxiety_qwalk_q...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.844198
9,event_predictions__convat_apex_anxiety_qwalk_q...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.451573


In [7]:
for name in ['calibration', 'event_summary', 'subject_summary', 'detail_files']:
    display(Markdown(f'## {name}'))
    display(outputs[name].head(20))


## calibration

,level,scenario,exp_name,label,mode,seed,aggregation,threshold,n,tn,...,accuracy,balanced_accuracy,rendah_precision,rendah_recall,rendah_f1,tinggi_precision,tinggi_recall,tinggi_f1,macro_f1,recall_gap
0,event_level_val,internal_val,convat_apex_anxiety_qholdout_q123_q4_q5_alis,qholdout_q123_q4_q5_alis,qholdout,11,top3_mean,0.925,550,164,...,0.661818,0.670208,0.573427,0.719298,0.638132,0.757576,0.621118,0.682594,0.660363,0.098180
1,event_level_val,internal_val,convat_apex_anxiety_qwalk_q12_q3_q4_alis,qwalk_q12_q3_q4_alis,qwalkforward,8,mean,0.675,550,188,...,0.669091,0.694701,0.564565,0.835556,0.673835,0.829493,0.553846,0.664207,0.669021,0.281709
2,event_level_val,internal_val,convat_apex_anxiety_qwalk_q123_q4_q5_alis,qwalk_q123_q4_q5_alis,qwalkforward,11,top3_mean,0.925,550,164,...,0.661818,0.670208,0.573427,0.719298,0.638132,0.757576,0.621118,0.682594,0.660363,0.098180


## event_summary

,level,scenario,exp_name,label,mode,seed,aggregation,threshold,n,tn,...,accuracy,balanced_accuracy,rendah_precision,rendah_recall,rendah_f1,tinggi_precision,tinggi_recall,tinggi_f1,macro_f1,recall_gap
0,event_level,15_only,convat_apex_anxiety_qholdout_q123_q4_q5_alis,qholdout_q123_q4_q5_alis,qholdout,11,top3_mean,0.925,1000,210,...,0.415000,0.517361,0.289655,0.750000,0.417910,0.745455,0.284722,0.412060,0.414985,0.465278
1,event_level,17_only,convat_apex_anxiety_qholdout_q123_q4_q5_alis,qholdout_q123_q4_q5_alis,qholdout,11,top3_mean,0.925,870,214,...,0.363218,0.421140,0.326718,0.654434,0.435845,0.474419,0.187845,0.269129,0.352487,0.466589
2,event_level,15_17_mixed,convat_apex_anxiety_qholdout_q123_q4_q5_alis,qholdout_q123_q4_q5_alis,qholdout,11,top3_mean,0.925,1870,424,...,0.390909,0.470795,0.307246,0.698517,0.426774,0.626531,0.243072,0.350257,0.388515,0.455445
3,event_level,15_only,convat_apex_anxiety_qwalk_q12_q3_q4_alis,qwalk_q12_q3_q4_alis,qwalkforward,8,mean,0.675,1000,195,...,0.407000,0.495437,0.277383,0.696429,0.396745,0.713805,0.294444,0.416912,0.406829,0.401984
4,event_level,17_only,convat_apex_anxiety_qwalk_q12_q3_q4_alis,qwalk_q12_q3_q4_alis,qwalkforward,8,mean,0.675,870,226,...,0.397701,0.456063,0.348228,0.691131,0.463115,0.542986,0.220994,0.314136,0.388625,0.470137
5,event_level,15_17_mixed,convat_apex_anxiety_qwalk_q12_q3_q4_alis,qwalk_q12_q3_q4_alis,qwalkforward,8,mean,0.675,1870,421,...,0.402674,0.478221,0.311391,0.693575,0.429811,0.640927,0.262866,0.372824,0.401318,0.430709
6,event_level,15_only,convat_apex_anxiety_qwalk_q123_q4_q5_alis,qwalk_q123_q4_q5_alis,qwalkforward,11,top3_mean,0.925,1000,210,...,0.415000,0.517361,0.289655,0.750000,0.417910,0.745455,0.284722,0.412060,0.414985,0.465278
7,event_level,17_only,convat_apex_anxiety_qwalk_q123_q4_q5_alis,qwalk_q123_q4_q5_alis,qwalkforward,11,top3_mean,0.925,870,214,...,0.363218,0.421140,0.326718,0.654434,0.435845,0.474419,0.187845,0.269129,0.352487,0.466589
8,event_level,15_17_mixed,convat_apex_anxiety_qwalk_q123_q4_q5_alis,qwalk_q123_q4_q5_alis,qwalkforward,11,top3_mean,0.925,1870,424,...,0.390909,0.470795,0.307246,0.698517,0.426774,0.626531,0.243072,0.350257,0.388515,0.455445


## subject_summary

,level,scenario,exp_name,label,mode,seed,aggregation,threshold,n,tn,...,balanced_accuracy,rendah_precision,rendah_recall,rendah_f1,tinggi_precision,tinggi_recall,tinggi_f1,macro_f1,recall_gap,event_threshold
0,subject_level,15_only,convat_apex_anxiety_qholdout_q123_q4_q5_alis,qholdout_q123_q4_q5_alis,qholdout,11,top3_mean->median,0.825,51,11,...,0.582046,0.323529,0.785714,0.458333,0.823529,0.378378,0.518519,0.488426,0.407336,0.925
1,subject_level,17_only,convat_apex_anxiety_qholdout_q123_q4_q5_alis,qholdout_q123_q4_q5_alis,qholdout,11,top3_mean->median,0.675,47,6,...,0.509804,0.375000,0.352941,0.363636,0.645161,0.666667,0.655738,0.509687,0.313725,0.925
2,subject_level,15_17_mixed,convat_apex_anxiety_qholdout_q123_q4_q5_alis,qholdout_q123_q4_q5_alis,qholdout,11,top3_mean->p90,0.825,98,2,...,0.517333,0.500000,0.064516,0.114286,0.691489,0.970149,0.807453,0.460870,0.905633,0.925
3,subject_level,15_only,convat_apex_anxiety_qwalk_q12_q3_q4_alis,qwalk_q12_q3_q4_alis,qwalkforward,8,mean->mean,0.575,51,9,...,0.564672,0.321429,0.642857,0.428571,0.782609,0.486486,0.600000,0.514286,0.156371,0.675
4,subject_level,17_only,convat_apex_anxiety_qwalk_q12_q3_q4_alis,qwalk_q12_q3_q4_alis,qwalkforward,8,mean->mean,0.475,47,6,...,0.526471,0.400000,0.352941,0.375000,0.656250,0.700000,0.677419,0.526210,0.347059,0.675
5,subject_level,15_17_mixed,convat_apex_anxiety_qwalk_q12_q3_q4_alis,qwalk_q12_q3_q4_alis,qwalkforward,8,mean->p90,0.700,98,10,...,0.519499,0.344828,0.322581,0.333333,0.695652,0.716418,0.705882,0.519608,0.393837,0.675
6,subject_level,15_only,convat_apex_anxiety_qwalk_q123_q4_q5_alis,qwalk_q123_q4_q5_alis,qwalkforward,11,top3_mean->median,0.825,51,11,...,0.582046,0.323529,0.785714,0.458333,0.823529,0.378378,0.518519,0.488426,0.407336,0.925
7,subject_level,17_only,convat_apex_anxiety_qwalk_q123_q4_q5_alis,qwalk_q123_q4_q5_alis,qwalkforward,11,top3_mean->median,0.675,47,6,...,0.509804,0.375000,0.352941,0.363636,0.645161,0.666667,0.655738,0.509687,0.313725,0.925
8,subject_level,15_17_mixed,convat_apex_anxiety_qwalk_q123_q4_q5_alis,qwalk_q123_q4_q5_alis,qwalkforward,11,top3_mean->p90,0.825,98,2,...,0.517333,0.500000,0.064516,0.114286,0.691489,0.970149,0.807453,0.460870,0.905633,0.925


## detail_files

,kind,scenario,path,size_mb
0,frame_predictions,15_only,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,32.275018
1,event_predictions,15_only,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.464824
2,subject_predictions,15_only,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.013523
3,frame_predictions,17_only,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,24.858193
4,event_predictions,17_only,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.397228
5,subject_predictions,17_only,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.012269
6,frame_predictions,15_17_mixed,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,57.207383
7,event_predictions,15_17_mixed,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.868990
8,subject_predictions,15_17_mixed,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,0.025386
9,frame_predictions,15_only,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,32.166882


## 4. Best external results

Inspect best event-level and subject-level results after bias-reduced thresholding.
Prefer `balanced_accuracy` and `macro_f1` over raw accuracy because POCABS labels are class-imbalanced.


In [8]:
event_summary = outputs['event_summary'].copy()
display(event_summary.sort_values(['scenario', 'balanced_accuracy', 'macro_f1'], ascending=[True, False, False]).reset_index(drop=True))

subject_summary = outputs['subject_summary'].copy()
display(Markdown('## Subject-derived summary'))
display(subject_summary.sort_values(['scenario', 'balanced_accuracy', 'macro_f1'], ascending=[True, False, False]).reset_index(drop=True))


,level,scenario,exp_name,label,mode,seed,aggregation,threshold,n,tn,...,accuracy,balanced_accuracy,rendah_precision,rendah_recall,rendah_f1,tinggi_precision,tinggi_recall,tinggi_f1,macro_f1,recall_gap
0,event_level,15_17_mixed,convat_apex_anxiety_qwalk_q12_q3_q4_alis,qwalk_q12_q3_q4_alis,qwalkforward,8,mean,0.675,1870,421,...,0.402674,0.478221,0.311391,0.693575,0.429811,0.640927,0.262866,0.372824,0.401318,0.430709
1,event_level,15_17_mixed,convat_apex_anxiety_qholdout_q123_q4_q5_alis,qholdout_q123_q4_q5_alis,qholdout,11,top3_mean,0.925,1870,424,...,0.390909,0.470795,0.307246,0.698517,0.426774,0.626531,0.243072,0.350257,0.388515,0.455445
2,event_level,15_17_mixed,convat_apex_anxiety_qwalk_q123_q4_q5_alis,qwalk_q123_q4_q5_alis,qwalkforward,11,top3_mean,0.925,1870,424,...,0.390909,0.470795,0.307246,0.698517,0.426774,0.626531,0.243072,0.350257,0.388515,0.455445
3,event_level,15_only,convat_apex_anxiety_qholdout_q123_q4_q5_alis,qholdout_q123_q4_q5_alis,qholdout,11,top3_mean,0.925,1000,210,...,0.415000,0.517361,0.289655,0.750000,0.417910,0.745455,0.284722,0.412060,0.414985,0.465278
4,event_level,15_only,convat_apex_anxiety_qwalk_q123_q4_q5_alis,qwalk_q123_q4_q5_alis,qwalkforward,11,top3_mean,0.925,1000,210,...,0.415000,0.517361,0.289655,0.750000,0.417910,0.745455,0.284722,0.412060,0.414985,0.465278
5,event_level,15_only,convat_apex_anxiety_qwalk_q12_q3_q4_alis,qwalk_q12_q3_q4_alis,qwalkforward,8,mean,0.675,1000,195,...,0.407000,0.495437,0.277383,0.696429,0.396745,0.713805,0.294444,0.416912,0.406829,0.401984
6,event_level,17_only,convat_apex_anxiety_qwalk_q12_q3_q4_alis,qwalk_q12_q3_q4_alis,qwalkforward,8,mean,0.675,870,226,...,0.397701,0.456063,0.348228,0.691131,0.463115,0.542986,0.220994,0.314136,0.388625,0.470137
7,event_level,17_only,convat_apex_anxiety_qholdout_q123_q4_q5_alis,qholdout_q123_q4_q5_alis,qholdout,11,top3_mean,0.925,870,214,...,0.363218,0.421140,0.326718,0.654434,0.435845,0.474419,0.187845,0.269129,0.352487,0.466589
8,event_level,17_only,convat_apex_anxiety_qwalk_q123_q4_q5_alis,qwalk_q123_q4_q5_alis,qwalkforward,11,top3_mean,0.925,870,214,...,0.363218,0.421140,0.326718,0.654434,0.435845,0.474419,0.187845,0.269129,0.352487,0.466589


## Subject-derived summary

,level,scenario,exp_name,label,mode,seed,aggregation,threshold,n,tn,...,balanced_accuracy,rendah_precision,rendah_recall,rendah_f1,tinggi_precision,tinggi_recall,tinggi_f1,macro_f1,recall_gap,event_threshold
0,subject_level,15_17_mixed,convat_apex_anxiety_qwalk_q12_q3_q4_alis,qwalk_q12_q3_q4_alis,qwalkforward,8,mean->p90,0.700,98,10,...,0.519499,0.344828,0.322581,0.333333,0.695652,0.716418,0.705882,0.519608,0.393837,0.675
1,subject_level,15_17_mixed,convat_apex_anxiety_qholdout_q123_q4_q5_alis,qholdout_q123_q4_q5_alis,qholdout,11,top3_mean->p90,0.825,98,2,...,0.517333,0.500000,0.064516,0.114286,0.691489,0.970149,0.807453,0.460870,0.905633,0.925
2,subject_level,15_17_mixed,convat_apex_anxiety_qwalk_q123_q4_q5_alis,qwalk_q123_q4_q5_alis,qwalkforward,11,top3_mean->p90,0.825,98,2,...,0.517333,0.500000,0.064516,0.114286,0.691489,0.970149,0.807453,0.460870,0.905633,0.925
3,subject_level,15_only,convat_apex_anxiety_qholdout_q123_q4_q5_alis,qholdout_q123_q4_q5_alis,qholdout,11,top3_mean->median,0.825,51,11,...,0.582046,0.323529,0.785714,0.458333,0.823529,0.378378,0.518519,0.488426,0.407336,0.925
4,subject_level,15_only,convat_apex_anxiety_qwalk_q123_q4_q5_alis,qwalk_q123_q4_q5_alis,qwalkforward,11,top3_mean->median,0.825,51,11,...,0.582046,0.323529,0.785714,0.458333,0.823529,0.378378,0.518519,0.488426,0.407336,0.925
5,subject_level,15_only,convat_apex_anxiety_qwalk_q12_q3_q4_alis,qwalk_q12_q3_q4_alis,qwalkforward,8,mean->mean,0.575,51,9,...,0.564672,0.321429,0.642857,0.428571,0.782609,0.486486,0.600000,0.514286,0.156371,0.675
6,subject_level,17_only,convat_apex_anxiety_qwalk_q12_q3_q4_alis,qwalk_q12_q3_q4_alis,qwalkforward,8,mean->mean,0.475,47,6,...,0.526471,0.400000,0.352941,0.375000,0.656250,0.700000,0.677419,0.526210,0.347059,0.675
7,subject_level,17_only,convat_apex_anxiety_qholdout_q123_q4_q5_alis,qholdout_q123_q4_q5_alis,qholdout,11,top3_mean->median,0.675,47,6,...,0.509804,0.375000,0.352941,0.363636,0.645161,0.666667,0.655738,0.509687,0.313725,0.925
8,subject_level,17_only,convat_apex_anxiety_qwalk_q123_q4_q5_alis,qwalk_q123_q4_q5_alis,qwalkforward,11,top3_mean->median,0.675,47,6,...,0.509804,0.375000,0.352941,0.363636,0.645161,0.666667,0.655738,0.509687,0.313725,0.925
